In [2]:
import networkx as nx
import matplotlib.pyplot as plt
import yaml
import os
#erc-karst package to pip install
import karstnet as kn # Karstnet-ERC version (pip install .)
import kntools as kt


## Load database

In [3]:
#load graph
cavename = 'Migovec'

#extract metadata from google sheet
metadata = kt.get_metadata(cavename,
                           url= 'https://docs.google.com/spreadsheets/d/' + 
                                '1I4avyKngrlN80VJwsUxC0phgaPiC1lAOujsaHYxTCEs' +
                                '/export?gid=1883705375&format=csv')

# kt.update_metadata_git(cavename)

In [3]:
progress_folder = 'c:/Users/celia/OneDrive - unine.ch/network_data_working_folder/2_in_progress'

#find automatically the number associated with the cavename
list_folder = os.listdir('../data')
# number = [item.split('_')[0] for item in list_folder if cavename in item][0]

# Load the SQL database
# path_dataset = f'../data/{number}_{cavename}/sql_database/{cavename}.sql'
path_dataset = f'{progress_folder}/{metadata['number']}_{cavename}/original_data/sql_data/{cavename}.sql'

# Create graph (do not remove flagged edges yet)
print('Loading therion sql data')
G  = kn.io_func.networkx_from_therion_sql(path_dataset) 

# Add metadata to graph
G.graph = metadata

Loading therion sql data
Therion Import -- Importing all links (including splays) -- 1.1640210151672363s
Therion Import -- Importing all nodes data (including splays) -- 1.1990840435028076s
Therion Import -- Create initial graph with all the data points (including splays) -- 1.351837396621704s
Therion Import -- Combine Stations with identical x,y,z -- 1.569610834121704s
0/7294 unique positions
1000/7294 unique positions
2000/7294 unique positions
3000/7294 unique positions
4000/7294 unique positions
5000/7294 unique positions
6000/7294 unique positions
7000/7294 unique positions
Therion Import -- Rename nodes -- 24.900851726531982s
0/7294 nodes to rename
1000/7294 nodes to rename
2000/7294 nodes to rename
3000/7294 nodes to rename
4000/7294 nodes to rename
5000/7294 nodes to rename
6000/7294 nodes to rename
7000/7294 nodes to rename
Therion Import -- concatenate old ic in a dictionnary -- 25.05287790298462s
Therion Import -- Relabel nodes -- 25.056541442871094s
Therion Import -- remove

## Add conduit geometry

In [1]:
# ADD GEOMETRY
#################

# !! Modify function to add both dimension from splays and lrud...

# kt.full_add_geom(G,cavename, metadata)

if metadata['original_format'].lower() in ('toporobot','vtopo', 'compass','ghtopo', 'shp_compass', 'unknown_lrud'):
    csdim_from_th = f'{progress_folder}/{metadata['number']}_{cavename}/original_data/therion_project/{cavename}_convert.th'
    print(f'extracting geometry data from :{ csdim_from_th}')
else:
    print('no lrud data to download from .th file')
    csdim_from_th = None

#todo:
#1. normalize vector when calculating main direction -- done 30 july 2024 -- but it change the plan and made it all perpendicular
#2. solve toporobot exception for vertical edges when calculating lrud_splays
#3. rename splay into splays_lrud and splays_shot
#4. calculate splays_lrud for therion data with splays (to see how they do)

#calculate geometry based on splays
#geommean_radius, cs_width, cs_height, projected_splays_u, projected_splays_v = ux.calc_conduit_dimensions(G, output = 'all')
print('Extracting dimension from SQL')
cs_dimension = kn.geom.calc_conduit_dimensions(G, output = 'csdim')
cs_dimension = {k: v for k, v in cs_dimension.items() if not (v[0] == 0 and v[1] == 0)}

if csdim_from_th is not None:
    print('full_sql_import -- Extracting dimension from therion file')
    cs_dimension_lrud = kn.geom.calc_csdim_from_th(G, csdim_from_th,metadata['units'])
    #remove node with 0 with and 0 height:
    cs_dimension_lrud = {k: v for k, v in cs_dimension_lrud.items() if not (v[0] == 0 and v[1] == 0)}
    nx.set_node_attributes(G, cs_dimension_lrud, 'csdim')  
#calcuate geomtery based on LRUD

#ADD GEOMETRY INFO ON THE NODES
#cross-sectional dimensions: width and height in the cross-sectional plan of the conduit perpendicular to the conduit direction
#nx.set_node_attributes(G, geommean_radius, 'geommean_radius') 
nx.set_node_attributes(G, cs_dimension, 'csdim') 

#exemple PLOT WITH WEIGHTED NODE GEOMETRY
node_sizes = []
labels = {}
for n in G.nodes:
    if G.nodes('csdim')[n] is not None:
        node_sizes.append( nx.get_node_attributes(G,'csdim')[n][0]*10 )
    else:
        node_sizes.append( 0.0 )

plt.figure()
ax = plt.gca()
ax.set_title(f'{cavename} - conduit dimension', loc='left',ha='left', wrap=True )
nx.draw(G,kn.utils.get_pos2d(G),node_size=node_sizes ,with_labels=False, node_color='red', ax=ax)

NameError: name 'metadata' is not defined

In [5]:
# CLEAN GRAPH
#################
# If H is not fully cleaned up, update the yaml file and re-run the code

correction_manual='c:/Users/celia/OneDrive - unine.ch/network_data_working_folder/2_in_progress/caves_info.yaml'
correction_auto=None

# 7. CLEAN GRAPH
#################
# If H is not fully cleaned up, update the yaml file and re-run the code
print('full_sql_import -- Cleaning graph')
H = G.copy()

with open(correction_manual) as f:
            yml = yaml.safe_load(f)

if cavename in yml.keys():

    if 'edge_flags' in yml[cavename].keys():
        kn.clean.flag_edges(H, yml[cavename]['edge_flags'], dict_address = nx.get_node_attributes(H,'fulladdress'))
    if 'node_flags' in yml[cavename].keys():
        kn.clean.flag_nodes(H,yml[cavename]['node_flags'], dict_address = nx.get_node_attributes(H,'fulladdress'))

if correction_auto:

    with open(correction_manual) as f:
                yml = yaml.safe_load(f)

    if 'edge_flags' in yml.keys():
        kn.clean.flag_edges(H, yml[cavename]['edge_flags'], dict_address = nx.get_node_attributes(H,'fulladdress'))
    if 'node_flags' in yml.keys():
        kn.clean.flag_nodes(H,yml[cavename]['node_flags'], dict_address = nx.get_node_attributes(H,'fulladdress'))

else:
    print('full_clean_G --- there is no correction in yaml file for this cave')

        # at this step, we remove the flagged edges (duplicate, surface, etc...)
print( 'There is ', nx.number_connected_components(H), 'connected components in the graph with additional edges')


I = H.copy()
kn.clean.remove_flagged_edges(I)

full_sql_import -- Cleaning graph
flag_edges - adding manual edges flags: dict_keys(['add', 'rmv'])
add
rmv
full_clean_G --- there is no correction in yaml file for this cave
There is  1 connected components in the graph with additional edges


In [ ]:
# metadata = kt.get_metadata(cavename)


# define path were the clean outpus will be saved
path_working_folder_outputs = f'c:/Users/celia/OneDrive - unine.ch/network_data_working_folder/2_in_progress/{metadata['number']}_{cavename}/outputs'

#save to github
path_public=f'C:/Users/celia/github/erc-karst-repositories/KNdata-public/data/{metadata['number']}_{cavename}'
path_confidential= f'C:/Users/celia/github/erc-karst-repositories/KNdata-confidential/data/{metadata['number']}_{cavename}'
path_secret= f'C:/Users/celia/github/erc-karst-repositories/KNdata-secret/data/{metadata['number']}_{cavename}'

if metadata['number'][0] == 'C':
    path_to_github = path_confidential
elif metadata['number'][0] == 'S':
        path_to_github = path_secret
else:
    path_to_github = path_public


print('full_save_outputs -- Saving graph in yaml format and csv format to working folder and github')

#save to working folder
#----------------------------
kn.io_func.nx2aven3d(G, I,  kn.utils.make_filepath(path_working_folder_outputs,'visualization/therion_cc_3d'), cavename)
kn.io_func.nx2clean3d(H,kn.utils.make_filepath(path_working_folder_outputs,'visualization/clean_therion_project'),cavename)
kn.io_func.networkx_to_yaml(I,f"{kn.utils.make_filepath(path_working_folder_outputs,'clean_graph_yaml')}/{cavename}.yaml")
kn.io_func.save_attribrutes_df_to_csv(I, path_working_folder_outputs, cavename=cavename)


# save outputs to github
#---------------------------
print('saving output to github')
kn.io_func.nx2clean3d(H, kn.utils.make_filepath(path_to_github,'visualization'),cavename)
kn.io_func.networkx_to_yaml(I, f"{kn.utils.make_filepath(path_to_github,'clean_data')}/{cavename}.yaml")
kn.io_func.save_attribrutes_df_to_csv(I, kn.utils.make_filepath(path_to_github,'clean_data'), cavename=cavename)
nx.write_sparse6(I, f'{kn.utils.make_filepath(path_to_github,'clean_data')}/{cavename}.s6')

# # save metadata to yaml
# with open(f'{path_to_github}/metadata.yaml', 'w') as file:
#     yaml.dump(metadata, file, default_flow_style=False, sort_keys=False, allow_unicode= True)

full_save_outputs -- Saving graph in yaml format and csv format to working folder and github
There is  1 connected components in the original graph
There is  1 connected components in the graph without flagged edges
There is no disconnected components, no need to merge
therion "c:/Users/celia/OneDrive - unine.ch/network_data_working_folder/2_in_progress/002_Migovec/outputs/visualization/therion_cc_3d/visualization/connectected_components_3d/Migovec_CC3d.thconfig"
running nx2clean3d
saving th files here: c:/Users/celia/OneDrive - unine.ch/network_data_working_folder/2_in_progress/002_Migovec/outputs/visualization/clean_therion_project/Migovec_clean.th
therion "c:/Users/celia/OneDrive - unine.ch/network_data_working_folder/2_in_progress/002_Migovec/outputs/visualization/clean_therion_project/Migovec_clean.thconfig"
Graph saved to yaml here: c:/Users/celia/OneDrive - unine.ch/network_data_working_folder/2_in_progress/002_Migovec/outputs/clean_graph_yaml//Migovec.yaml
saved edge list to  c